In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('utils.py')))
import pandas as pd
import numpy as np
from utils import (
    get_data_splits,
    char_ngram_similarity,
    get_char_ngram_features,
)

## Character N-grams for Robust Similarity

While word-level features (Joel) and TF-IDF (Carlos) are powerful, they are sensitive to spelling variations and typos. **Character N-grams** decompose words into smaller chunks, making the similarity measure more robust.

For example, "python" and "pyton" share zero words but share many character 3-grams (*pyt*, *yth*, *tho*, *hon* vs *pyt*, *yto*, *ton*).

### Implementation Details
- **Feature**: Character-level Jaccard similarity using 3-grams.
- **Algorithm**: We extract all unique character 3-grams from both strings and compute the intersection over union.
- **Advantage**: Captures morphological similarities and handles typos better than word-level models.

In [2]:
q1 = "How do I learn Python?"
q2 = "How do I lern Pyton?"  # Typos
q3 = "How do I cook an omelette?"

print(f"Similarity (q1, q2): {char_ngram_similarity(q1, q2):.4f}")
print(f"Similarity (q1, q3): {char_ngram_similarity(q1, q3):.4f}")

Similarity (q1, q2): 0.5833
Similarity (q1, q3): 0.1892


## Distribution on a sample

We evaluate how well this feature separates duplicate questions from non-duplicates.

In [3]:
DATA_PATH = "quora_data.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join(os.path.expanduser("~"), "Datasets", "QuoraQuestionPairs", "quora_data.csv")

quora_df = pd.read_csv(DATA_PATH)
train_df, _, _ = get_data_splits(quora_df)

sample = train_df.sample(500, random_state=42)
X_char = get_char_ngram_features(sample)

dup_mask = sample["is_duplicate"].values.astype(bool)
print(f"Mean Char 3-gram Similarity for duplicates:     {X_char[dup_mask].mean():.3f}")
print(f"Mean Char 3-gram Similarity for non-duplicates: {X_char[~dup_mask].mean():.3f}")

Mean Char 3-gram Similarity for duplicates:     0.464
Mean Char 3-gram Similarity for non-duplicates: 0.297
